[Start](../README.md) · [Demos](README.md) · [Nächste Demo: REST-APIs](02_rest_apis_coingecko_flickr.ipynb)

# Demo 1 · Strukturierte Daten: Wechselkurse im XML-Format

**Lernziele:** Du kannst einen strukturierten Download von HTML-Scraping unterscheiden, XML-Namensräume verwenden, Felder fachlich prüfen und einen Export mit Herkunftsinformationen erstellen.

Die Europäische Zentralbank (EZB/ECB) stellt Euro-Referenzkurse als XML bereit. Eine HTML-Tabelle auf ihrer Website auszulesen wäre dafür ein unnötiger Umweg. Ein strukturierter Download eignet sich ebenfalls für Web-Ingestion – auch ohne klassische REST-API.

**Standard: synthetische Beispieldaten.** Alle mitgelieferten Kurse und das Beispieldatum sind erfunden. Der Standardablauf benötigt nach der Einrichtung kein Internet und keinen Browsercontainer. Erst `EZB_LIVE = True` veranlasst einen einzelnen Live-Abruf. Es gibt bei Live-Fehlern keinen automatischen Wechsel zu Beispieldaten.

## 1 · Umgebung und Datenquelle wählen

Wähle den Kernel **Python (webscraping-workshop)** und führe die Zellen der Reihe nach aus. Der vollständige Einrichtungscheck steht in [Notebook 00](../notebooks/00_setup_check.ipynb).

In [1]:
from pathlib import Path
import sys

# Funktioniert im Repo-Hauptordner und in dessen Unterordnern.
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Öffne das Notebook innerhalb des Workshop-Repos.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repo gefunden:", ROOT.name)

Repo gefunden: webscraping-workshop-2026


In [2]:
import json
import math
from datetime import date, datetime, timezone

import pandas as pd
import requests
from lxml import etree
from IPython.display import display

EZB_LIVE = False
EZB_URL = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-daily.xml"
EZB_SAMPLE = ROOT / "data" / "samples" / "ezb_sample.xml"
source_mode = "live" if EZB_LIVE else "sample"
print("Datenmodus:", source_mode, "– SYNTHETISCH" if not EZB_LIVE else "– externer Abruf")

Datenmodus: sample – SYNTHETISCH


## 2 · Daten beziehen

Beim Live-Abruf begrenzen wir die Wartezeit und prüfen den HTTP-Status. Ein erfolgreicher HTTP-Abruf allein beweist noch nicht, dass die Antwort das erwartete XML enthält: Diese Prüfung folgt beim Parsen.

`acquired_at_utc` beschreibt unseren Zugriff auf die Datei bzw. den Webdienst. **Das Kursdatum steht separat im XML.** An Wochenenden und Feiertagen ist das jüngste verfügbare Datum nicht zwingend das heutige Datum.

In [3]:
if EZB_LIVE:
    try:
        with requests.get(EZB_URL, timeout=(5, 20), allow_redirects=False) as response:
            if response.status_code != 200:
                raise RuntimeError(f"EZB: HTTP {response.status_code}; Abruf wurde beendet.")
            xml_bytes = response.content
    except requests.RequestException:
        raise RuntimeError("EZB: Verbindung fehlgeschlagen. Prüfe Netzwerk und Erreichbarkeit.") from None
    source_location = EZB_URL
else:
    xml_bytes = EZB_SAMPLE.read_bytes()
    source_location = "data/samples/ezb_sample.xml"

acquired_at_utc = datetime.now(timezone.utc).isoformat()
print(f"{len(xml_bytes)} Bytes geladen; Modus: {source_mode}.")

669 Bytes geladen; Modus: sample.


## 3 · XML mit Namensräumen auslesen

XML-Elemente werden durch **Namensraum und lokalen Namen** identifiziert. Die Elemente heissen hier beispielsweise nicht nur `Cube`, sondern gehören zum EZB-Namensraum. Unser frei gewähltes Präfix `ecb` bindet diesen Namensraum an die Suche.

Die Struktur lautet: `Envelope → Cube → Cube mit Datum → Cube je Währung`. Für den täglichen Download erwarten wir genau einen Datumsblock und mindestens einen Wechselkurs. Wir prüfen ausserdem Währungscodes, positive endliche Zahlen und doppelte Währungen. Bei einem geänderten Schema stoppen wir mit einer erklärenden Meldung.

In [4]:
def parse_ezb(xml_content):
    parser = etree.XMLParser(resolve_entities=False, no_network=True)
    try:
        document = etree.fromstring(xml_content, parser=parser)
    except etree.XMLSyntaxError:
        raise ValueError("Die Antwort ist kein gültiges XML.") from None

    ns = {"ecb": "http://www.ecb.int/vocabulary/2002-08-01/eurofxref"}
    days = document.findall("ecb:Cube/ecb:Cube", namespaces=ns)
    if len(days) != 1 or days[0].get("time") is None:
        raise ValueError("Erwartet wird genau ein EZB-Datumsblock mit Attribut time.")
    try:
        rate_date = date.fromisoformat(days[0].get("time")).isoformat()
    except ValueError:
        raise ValueError("Das Kursdatum hat nicht das erwartete ISO-Format YYYY-MM-DD.") from None

    rows = []
    for item in days[0].findall("ecb:Cube", namespaces=ns):
        currency = item.get("currency", "")
        if len(currency) != 3 or not currency.isascii() or not currency.isalpha() or not currency.isupper():
            raise ValueError("Ein Währungscode fehlt oder hat ein unerwartetes Format.")
        try:
            rate = float(item.get("rate", ""))
        except (TypeError, ValueError):
            raise ValueError("Ein Wechselkurs fehlt oder ist nicht numerisch.") from None
        if not math.isfinite(rate) or rate <= 0:
            raise ValueError("Wechselkurse müssen positiv und endlich sein.")
        rows.append({"rate_date": rate_date, "base_currency": "EUR",
                     "currency": currency, "rate_per_eur": rate})

    if not rows:
        raise ValueError("Keine Kurse gefunden. Prüfe Schema und Namensraum.")
    result = pd.DataFrame(rows)
    if result["currency"].duplicated().any():
        raise ValueError("Doppelte Währung im täglichen Datenbestand.")
    return result.sort_values("currency").reset_index(drop=True)

rates = parse_ezb(xml_bytes)
rates["source_mode"] = source_mode
rates["acquired_at_utc"] = acquired_at_utc
display(rates)

,rate_date,base_currency,currency,rate_per_eur,source_mode,acquired_at_utc
0,2026-09-01,EUR,CHF,0.96,sample,2026-09-30T10:57:59.727957+00:00
1,2026-09-01,EUR,GBP,0.85,sample,2026-09-30T10:57:59.727957+00:00
2,2026-09-01,EUR,JPY,160.00,sample,2026-09-30T10:57:59.727957+00:00
3,2026-09-01,EUR,USD,1.12,sample,2026-09-30T10:57:59.727957+00:00


## 4 · Die Richtung des Kurses verstehen

Die **Basiswährung ist EUR**: `rate_per_eur = 0.96` für CHF bedeutet im synthetischen Beispiel **1 EUR = 0.96 CHF**. Der Kehrwert liefert EUR pro CHF. Aus der Höhe zweier unterschiedlich denominierter Kurse lässt sich keine Rangfolge «starker» Währungen ableiten.

Die folgende Umrechnung dient ausschliesslich dazu, die Einheit nachzuvollziehen. Live-Referenzkurse sind keine verbindlichen An- oder Verkaufskurse für eine Transaktion.

In [5]:
chf_rows = rates.loc[rates["currency"].eq("CHF"), "rate_per_eur"]
if len(chf_rows) != 1:
    raise ValueError("Für die Umrechnungsdemo wird genau ein CHF-Kurs benötigt.")
chf_per_eur = float(chf_rows.iloc[0])
amount_eur = 100.0
amount_chf = amount_eur * chf_per_eur
print(f"{amount_eur:.2f} EUR entsprechen {amount_chf:.2f} CHF im Modus {source_mode}.")
print(f"1 CHF entspricht rechnerisch {1 / chf_per_eur:.6f} EUR.")

100.00 EUR entsprechen 96.00 CHF im Modus sample.
1 CHF entspricht rechnerisch 1.041667 EUR.


## 5 · Rohdaten, Tabelle und Herkunft zusammen speichern

Drei Dateien haben unterschiedliche Aufgaben: `raw.xml` bewahrt die Eingabe, `rates.csv` erleichtert die Weiterverarbeitung und `metadata.json` beschreibt deren Herkunft. Die Pfade trennen **sample** und **live**. Wiederholtes Ausführen ersetzt jeweils den letzten Export desselben Modus.

Das Beispiel macht keine vollständige Archivierung vor. Für eine produktive Pipeline wären zusätzlich versionierte Ablagen und nachvollziehbare Läufe sinnvoll.

In [6]:
output_dir = ROOT / "data" / "output" / "demo01_ezb" / source_mode
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "raw.xml").write_bytes(xml_bytes)
rates.to_csv(output_dir / "rates.csv", index=False, encoding="utf-8")
metadata = {
    "provider": "ECB", "source_mode": source_mode,
    "synthetic": not EZB_LIVE, "source_location": source_location,
    "acquired_at_utc": acquired_at_utc,
    "rate_date": rates["rate_date"].iloc[0],
    "base_currency": "EUR", "rate_unit": "currency units per EUR",
    "row_count": len(rates),
}
(output_dir / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
roundtrip = pd.read_csv(output_dir / "rates.csv")
if len(roundtrip) != len(rates) or set(roundtrip["source_mode"]) != {source_mode}:
    raise RuntimeError("Exportprüfung fehlgeschlagen.")
print("Export geprüft:", output_dir.relative_to(ROOT))

Export geprüft: data\output\demo01_ezb\sample


## Reflexion und Transfer

1. Weshalb ist der XML-Download hier geeigneter als das Auslesen der sichtbaren HTML-Tabelle?
2. Worin unterscheiden sich **Kursdatum** und **Zeitpunkt unseres Datenbezugs**?
3. Was würde eine leere Währungstabelle bedeuten? Welche Prüfungen verhindern einen unbemerkten Schemafehler?
4. Wie würdest Du aus den EUR-basierten Werten CHF pro USD berechnen? Benenne Zähler und Nenner mit ihrer Einheit.
5. Welche zusätzliche Ablagestruktur wäre nötig, damit frühere Abrufe erhalten bleiben?

**Kontrollpunkt im Sample-Modus:** vier Währungen, Kursdatum `2026-09-01`, 100 EUR → 96 CHF; Export unter `data/output/demo01_ezb/sample/`. Alle Werte sind synthetisch.

## Quellen und Herkunft

- [EZB: Euro-Referenzkurse und Downloadangebote](https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/index.en.html)
- [EZB: tägliche XML-Datei](https://www.ecb.europa.eu/stats/eurofxref/eurofxref-daily.xml)
- [lxml: Tutorial zu XML und Namensräumen](https://lxml.de/tutorial.html)

Dokumentation und Endpunkte geprüft am **30.09.2026**. Das mitgelieferte XML ist eigens erstelltes synthetisches Lehrmaterial, kein archivierter EZB-Abruf. Der Anwendungsfall greift das bereitgestellte Vorjahresmaterial `BS_ECB_example.ipynb` auf. Die Materialsammlung enthält auf Gellrich, M. (2021), *Exercises in the theme block “Data Ingestion”*, ZHAW, zurückgehende Beispiele; diese Fassung wurde für den aktuellen Workshop neu ausgearbeitet.

[Zurück zu den Demos](README.md) · [Weiter: REST-APIs](02_rest_apis_coingecko_flickr.ipynb)